In [1]:
import pandas as pd

df = pd.read_csv('dataset.csv', usecols=['app_id', 'app_name', 'review_score'])

#top 20 po broju recenzija
top_apps_counts = df.groupby(['app_id', 'app_name']).size().reset_index(name='review_count')
top_20_apps = top_apps_counts.sort_values('review_count', ascending=False).head(20)

top_20_ids = top_20_apps['app_id'].tolist()
df_top20 = df[df['app_id'].isin(top_20_ids)]

#poz i neg recenzije
sentiment_counts = df_top20.groupby(['app_id', 'app_name', 'review_score']).size().unstack(fill_value=0).reset_index()
sentiment_counts.columns = ['app_id', 'app_name', 'negativne', 'pozitivne']

result = top_20_apps.merge(sentiment_counts, on=['app_id', 'app_name'])

print("\nTop 20 igara sa pozitivnim i negativnim recenzijama:")
print(result[['app_name', 'review_count', 'pozitivne', 'negativne']])


Top 20 igara sa brojem pozitivnih i negativnih recenzija:
                    app_name  review_count  pozitivne  negativne
0                   PAYDAY 2         88973      61765      27208
1                       DayZ         88850      58847      30003
2                   Terraria         84828      82350       2478
3                       Rust         77037      61032      16005
4                     Dota 2         73541      62923      10618
5              Rocket League         54227      51120       3107
6                  Undertale         51918      49851       2067
7              Left 4 Dead 2         50980      47183       3797
8                   Warframe         48229      43749       4480
9         Grand Theft Auto V         42374      31355      11019
10                 Robocraft         41596      29040      12556
11                 Starbound         41141      36134       5007
12                  Portal 2         38924      38448        476
13           Space Engineers   

Извучено је 20 игара, јер је први покушај добијања топ 5 дао и Терарију (негативне < 5000). 
Даље, треба проверити смисленост рецензија за сваку од игрица, задајући одређени праг дужине рецензије.

In [6]:
df = pd.read_csv('dataset.csv', usecols=['app_name', 'review_score', 'review_text'])

top20_names = [
    'PAYDAY 2', 'DayZ', 'Terraria', 'Rust', 'Dota 2',
    'Rocket League', 'Undertale', 'Left 4 Dead 2', 'Warframe', 'Grand Theft Auto V',
    'Robocraft', 'Starbound', 'Portal 2', 'Space Engineers', 'Fallout: New Vegas',
    'Arma 3', 'The Witcher 3: Wild Hunt', 'Heroes & Generals', 'BioShock Infinite', 'The Forest'
]

df_top20 = df[df['app_name'].isin(top20_names)].copy()

df_top20['word_count'] = df_top20['review_text'].fillna('').str.split().str.len()

print("=" * 100)
print("Analiza recenzija (duzina >= 10 reci) za top 20 igara")
print("=" * 100)

candidates = []

for game in top20_names:
    game_df = df_top20[df_top20['app_name'] == game]
    total = len(game_df)
    
    long_df = game_df[game_df['word_count'] >= 10]
    pos_long = len(long_df[long_df['review_score'] == 1])
    neg_long = len(long_df[long_df['review_score'] == -1])
    
    meets_5000 = pos_long >= 5000 and neg_long >= 5000
    
    if pos_long >= 4000 and neg_long >= 4000:
        candidates.append((game, pos_long, neg_long, meets_5000))
    
    print(f"\n{game}:")
    print(f"  Ukupno recenzija: {total}")
    print(f"  Recenzije >= 10 reci: pozitivnih {pos_long}, negativnih {neg_long}")
    print(f"  Moze 5000/5000? {'DA' if meets_5000 else 'NE'}")

print('\n')
print("=" * 100)
print("KANDIDATI za balansiran skup (pozitivne i negativne > 4000):")
print("=" * 100)
for game, pos, neg, meets in candidates:
    print(f"{game}: poz={pos}, neg={neg} -> 5000/5000: {'DA' if meets else 'NE'}")

Analiza recenzija (duzina >= 10 reci) za top 20 igara

PAYDAY 2:
  Ukupno recenzija: 88973
  Recenzije >= 10 reci: pozitivnih 42302, negativnih 21692
  Moze 5000/5000? DA

DayZ:
  Ukupno recenzija: 88850
  Recenzije >= 10 reci: pozitivnih 0, negativnih 0
  Moze 5000/5000? NE

Terraria:
  Ukupno recenzija: 84828
  Recenzije >= 10 reci: pozitivnih 59397, negativnih 1763
  Moze 5000/5000? NE

Rust:
  Ukupno recenzija: 77037
  Recenzije >= 10 reci: pozitivnih 0, negativnih 1
  Moze 5000/5000? NE

Dota 2:
  Ukupno recenzija: 73541
  Recenzije >= 10 reci: pozitivnih 29403, negativnih 6984
  Moze 5000/5000? DA

Rocket League:
  Ukupno recenzija: 54227
  Recenzije >= 10 reci: pozitivnih 34343, negativnih 2439
  Moze 5000/5000? NE

Undertale:
  Ukupno recenzija: 51918
  Recenzije >= 10 reci: pozitivnih 39457, negativnih 1631
  Moze 5000/5000? NE

Left 4 Dead 2:
  Ukupno recenzija: 50980
  Recenzije >= 10 reci: pozitivnih 30751, negativnih 2843
  Moze 5000/5000? NE

Warframe:
  Ukupno recenzija:

На основу добијених података, консултовати се са асистентом поводом одабира преосталих игара.

-Након разговора са асистентом, одобрено је узимање игара са мање од 5000/5000 рецензија - игре узете су Warframe и Arma 3.

In [7]:
df = pd.read_csv('dataset.csv', usecols=['app_name', 'review_score', 'review_text'])

top5_games = ['PAYDAY 2', 'Dota 2', 'Grand Theft Auto V', 'Warframe', 'Arma 3']

df_filtered = df[df['app_name'].isin(top5_games)].copy()

df_filtered['word_count'] = df_filtered['review_text'].fillna('').str.split().str.len()
df_filtered = df_filtered[df_filtered['word_count'] >= 10]

targets = {
    'PAYDAY 2': 5000,
    'Dota 2': 5000,
    'Grand Theft Auto V': 5000,
    'Warframe': 3627,
    'Arma 3': 2960
}

final_dfs = []

for game, n in targets.items():
    game_df = df_filtered[df_filtered['app_name'] == game]
    
    pos = game_df[game_df['review_score'] == 1]
    neg = game_df[game_df['review_score'] == -1]
    
    pos_sample = pos.sample(n=min(len(pos), n), random_state=42)
    neg_sample = neg.sample(n=min(len(neg), n), random_state=42)
    
    balanced = pd.concat([pos_sample, neg_sample])
    final_dfs.append(balanced)

final_df = pd.concat(final_dfs, ignore_index=True)
final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)  #mesanje bitno zbog samplinga po batchevima

final_df.to_csv('steam_dataset_cleaned.csv', index=False)

Након извршавања скрипте и увида у фајл са чистим подацима, примећени су специјални карактери и ASCII art који уносе шум, те је потребно прочистити одабране рецензије од таквог садржаја.

In [12]:
import re

df = pd.read_csv('steam_dataset_cleaned.csv')

#print(f"Recenzija pre ciscenja: {len(df)}")

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'[^a-zA-Z0-9\s\.\,\!\?\/\-:]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['review_text'] = df['review_text'].apply(clean_text)
df = df[df['review_text'].str.strip() != '']
df['word_count'] = df['review_text'].str.split().str.len()

df = df[['app_name', 'review_score', 'review_text']]

df.to_csv('clean_steam_dataset.csv', index=False)

#print(f"Ostalo recenzija: {len(df)}")

Recenzija pre ciscenja: 43174
Ostalo recenzija: 43160
